# VVC_RL D-Series — Stage 2: Train 10 updates

Default **D4**: D3 state/Actor/reward with analytic critic budget decomposition. Fresh D4 training to update 10. Inspect validation before continuing.

Functional source commit: `c79f10308b2aded88ea8c48c88a75952a13e7f68`. GPU **T4 x2**, Internet ON. Same locked mini16 Train / 4 Validation. No Test use. Fresh D4 gate/checkpoints only; no D3/B/Combined resume.

Inputs may be intact SAFE ZIPs or Kaggle-expanded `VVC_RL_payload.bin` + `payload_sha256.txt`; toolchain ZIP or expanded manifest/executable directory. Do not manually expand the inner payload.bin.

In [ ]:
STAGE = 2
from pathlib import Path
import sys, os, json, time, subprocess, shutil
SOURCE_COMMIT = 'c79f10308b2aded88ea8c48c88a75952a13e7f68'
EXPECTED_SOURCE_SHA256 = 'e44a51fc01adcb986c53614fd596405328cbc40c080799d7f665129c5775577f'
REPOSITORY = 'https://github.com/youngan2810/VVC_RL.git'
VARIANTS = {
    'D0': ('encoder-feedback-v1','task-budget-v3-progressive-soft-iou'),
    'D1': ('d1-decoupled-v1','task-budget-v3-progressive-soft-iou'),
    'D1i': ('d1-isolated-v1','task-budget-v3-progressive-soft-iou'),
    'D2': ('d2-region-v1','task-budget-v3-progressive-soft-iou'),
    'D2-neutral': ('d2-region-neutral-v1','task-budget-v3-progressive-soft-iou'),
    'D3': ('d3-online-budget-v1','task-budget-v4-online-soft-iou'),
    'D4': ('d4-budget-value-v1','task-budget-v4-online-soft-iou'),
}
VARIANT = 'D4'  # Fresh run: analytic critic budget offset; same D3 Actor/reward
SEED = 20261120
FRAMES = 256
VAL_EVERY = 5
SESSION_HOURS = 11.0
# Optional: explicit mounted paths, if automatic discovery is ambiguous.
PREFLIGHT_INPUT = ''
TOOLCHAIN_INPUT = ''
TRAINING_INPUT = ''
# Only for continuing an interrupted stage; otherwise leave empty.
PARTIAL_STAGE_INPUT = ''
assert VARIANT in VARIANTS and 0 < SESSION_HOURS <= 11.75
STATE_VERSION, REWARD_VERSION = VARIANTS[VARIANT]
WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input')
assert WORK.is_dir() and INPUT.is_dir()
TAG=f'{VARIANT}_seed{SEED}_{SOURCE_COMMIT[:8]}'
BASE=WORK/('VVC_RL_D_'+SOURCE_COMMIT[:8]); REPO=BASE/'source'; TOOLS=BASE/'tools'
PREFLIGHT=WORK/f'preflight_{TAG}'
TRAIN=WORK/f'vvc_rl_{TAG}_stage{STAGE}'
GATE_NAME=f'VVC_RL_PREFLIGHT_{PREFLIGHT.name}_SAFE.zip'
TOOLS_NAME=f'VVC_RL_{TAG}_TOOLCHAIN.zip'
RUN10_NAME=f'VVC_RL_MINI_vvc_rl_{TAG}_stage2_SAFE.zip'
RUN40_NAME=f'VVC_RL_MINI_vvc_rl_{TAG}_stage3_SAFE.zip'
LOGS=WORK/f'logs_stage{STAGE}_{TAG}_{time.time_ns()}'; LOGS.mkdir(parents=True)
STARTED=time.monotonic()
os.environ['YOLOv5_AUTOINSTALL']='false'
print('STAGE',STAGE,VARIANT,STATE_VERSION,REWARD_VERSION)


In [ ]:
def live(args,name,cwd=None):
    args=list(map(str,args)); log=LOGS/f'{name}.log'
    print('$',*args,flush=True)
    with log.open('w',encoding='utf-8') as out:
        p=subprocess.Popen(args,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,errors='replace')
        try:
            for line in p.stdout:
                out.write(line);out.flush();print(line,end='',flush=True)
            if p.wait():raise RuntimeError(f'{name} failed; see {log}')
        finally:
            if p.poll() is None:p.terminate();p.wait()
    return log

def common_args():
    return ['--toolchain',TOOLS/'TOOLCHAIN.json','--tools',TOOLS,'--input',INPUT,
        '--frames',FRAMES,'--state-version',STATE_VERSION,'--reward-version',REWARD_VERSION,
        '--rate-tolerance','0.02','--gamma','1.0','--gae-lambda','0.95','--train-schedule','mixed','--seed',SEED]

def verify_source():
    script="""import sys,subprocess
from vvcrl.repository import verify_checkout
from vvcrl.config import fingerprint
assert subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()==sys.argv[1]
print('TRACKED_RAW_BYTES_PASS',verify_checkout('.'))
# No untracked functional input may silently change the source fingerprint.
assert not subprocess.check_output(['git','ls-files','--others','--exclude-standard','--','vvcrl','native','vendor','configs','scripts','tests'],text=True).strip()
assert fingerprint('.')==sys.argv[2], 'Functional source differs from this notebook'
print('FUNCTIONAL_SOURCE_SHA256',fingerprint('.'))
"""
    live([sys.executable,'-c',script,SOURCE_COMMIT,EXPECTED_SOURCE_SHA256],'source_verify',REPO)

def gate_check():
    script="""from pathlib import Path
import json,sys
from vvcrl.gates import binding,require_gate
from vvcrl.config import Profile,PPOConfig,RewardConfig
p,t,sv,rv,frames=sys.argv[1:]
b=binding(Path.cwd(),json.loads(Path(t).read_text()),Profile(frames=int(frames)),2,RewardConfig(rv,.02),PPOConfig(),'mixed',sv)
print('GATE_RECHECK_PASS',require_gate(p,b)['status'])
"""
    live([sys.executable,'-c',script,PREFLIGHT,TOOLS/'TOOLCHAIN.json',STATE_VERSION,REWARD_VERSION,FRAMES],'gate_recheck',REPO)

import zipfile, hashlib, tempfile
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()

def safe_package(path):
    p=Path(path)
    if p.is_dir():
        if all((p/k).is_file() for k in ('manifest.json','encoder','vanilla','decoder')):return p
        p=p/'VVC_RL_payload.bin'
    assert p.is_file(),f'Missing package: {p}'
    if p.suffix=='.bin':
        side=p.parent/'payload_sha256.txt'
        assert side.is_file(),f'Keep payload_sha256.txt beside {p.name}'
        assert sha256(p)==side.read_text().strip(),'Raw SAFE payload hash mismatch'
    return p

def locate(name,explicit=''):
    if explicit:return safe_package(explicit)
    hits=sorted(p for p in INPUT.rglob(name) if p.is_file())
    if len(hits)==1:return safe_package(hits[0])
    if len(hits)>1:raise RuntimeError(f'Ambiguous {name}: set its exact path')
    # Kaggle expands the outer SAFE ZIP to payload.bin + sidecar; it may also
    # expand the toolchain ZIP to three executables + manifest.json.
    candidates={}
    if name==TOOLS_NAME:
        for p in sorted(INPUT.rglob('manifest.json')):
            try:
                m=json.loads(p.read_text())
                if m.get('source_commit')!=SOURCE_COMMIT or m.get('binding',{}).get('state_version')!=STATE_VERSION:continue
                if m['binding']['source_sha256']!=EXPECTED_SOURCE_SHA256:continue
                if not all((p.parent/k).is_file() for k in ('encoder','vanilla','decoder')):continue
                candidates.setdefault(json.dumps(m,sort_keys=True),p.parent)
            except (ValueError,KeyError,OSError):continue
    else:
        for p in sorted(INPUT.rglob('VVC_RL_payload.bin')):
            try:
                side=p.parent/'payload_sha256.txt'
                if not side.is_file():continue
                with zipfile.ZipFile(p) as z:
                    if name==GATE_NAME:
                        proof=json.loads(z.read('PREFLIGHT_GATE.json'));b=proof['binding']
                        if proof['status']!='PASS_READY_FOR_MINI':continue
                    else:
                        b=json.loads(z.read('TRAIN_BINDING.json'));progress=json.loads(z.read('PROGRESS.json'))
                        if progress['iteration']!=(10 if name==RUN10_NAME else 40):continue
                    if b.get('source_sha256')!=EXPECTED_SOURCE_SHA256 or b.get('state_version')!=STATE_VERSION:continue
                    if b.get('reward')!={'version':REWARD_VERSION,'rate_tolerance':.02}:continue
                candidates.setdefault(side.read_text().strip(),p)
            except (ValueError,KeyError,OSError,zipfile.BadZipFile):continue
    if len(candidates)==1:return safe_package(next(iter(candidates.values())))
    if len(candidates)>1:raise RuntimeError(f'Multiple different matching {name}: set explicit path')
    found=[]
    for wrapper in sorted(INPUT.rglob('_output_.zip')):
        with zipfile.ZipFile(wrapper) as z:
            for info in z.infolist():
                if not info.is_dir() and Path(info.filename).name==name:found.append((wrapper,info.filename))
    if len(found)!=1:raise RuntimeError(f'Missing {name}. Attach the NEW D4 stage output (ZIP or payload+sidecar); found wrappers: {found}')
    wrapper,member=found[0];folder=BASE/'mounted_packages';folder.mkdir(parents=True,exist_ok=True)
    p=folder/name;tmp=p.with_suffix('.partial')
    with zipfile.ZipFile(wrapper) as z,z.open(member) as src,tmp.open('wb') as dst:shutil.copyfileobj(src,dst,1024*1024)
    tmp.replace(p);return safe_package(p)

def import_safe(package,destination,label):
    assert not destination.exists() or not any(destination.iterdir()), f'Nonempty {destination}: use a fresh session'
    live([sys.executable,'-u','-m','vvcrl','import','--output',destination,'--package',package],label,REPO)

def save_toolchain():
    gate=json.loads((PREFLIGHT/'PREFLIGHT_GATE.json').read_text())
    tool=json.loads((TOOLS/'TOOLCHAIN.json').read_text())
    hashes=gate['binding']['binary_sha256']
    assert all(sha256(tool[k])==hashes[k] for k in ('encoder','vanilla','decoder'))
    target=WORK/TOOLS_NAME;tmp=target.with_suffix('.partial')
    with zipfile.ZipFile(tmp,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=3) as z:
        z.writestr('manifest.json',json.dumps({'source_commit':SOURCE_COMMIT,'binding':gate['binding']}))
        for k in ('encoder','vanilla','decoder'):z.write(tool[k],k)
    tmp.replace(target)
    print('TOOLCHAIN_READY',target)

def restore_toolchain(package):
    gate=json.loads((PREFLIGHT/'PREFLIGHT_GATE.json').read_text())
    TOOLS.mkdir(parents=True,exist_ok=True)
    binaries={}
    package=Path(package)
    z=None if package.is_dir() else zipfile.ZipFile(package)
    if z is None:
        assert all((package/k).is_file() for k in ('manifest.json','encoder','vanilla','decoder'))
        read=lambda n:(package/n).read_bytes()
    else:
        assert sorted(z.namelist())==['decoder','encoder','manifest.json','vanilla']
        read=z.read
    try:
        m=json.loads(read('manifest.json'))
        assert m['source_commit']==SOURCE_COMMIT and m['binding']==gate['binding']
        for k in ('encoder','vanilla','decoder'):
            p=TOOLS/k;tmp=p.with_suffix('.partial');tmp.write_bytes(read(k))
            assert sha256(tmp)==gate['binding']['binary_sha256'][k],k
            tmp.chmod(0o755);tmp.replace(p);binaries[k]=str(p)
    finally:
        if z is not None:z.close()
    (TOOLS/'TOOLCHAIN.json').write_text(json.dumps(binaries,indent=2))
    # Install the same dependency set as setup_kaggle.py, but NEVER rebuild the
    # native binaries. Restore pinned detector code and verified weights.
    live(['apt-get','update','-qq'],'apt_update')
    live(['apt-get','install','-y','-qq','cmake','g++','ffmpeg','pkg-config','libswscale-dev','libavutil-dev'],'apt_deps')
    live([sys.executable,'-m','pip','install','--disable-pip-version-check','setuptools==80.9.0','gitpython==3.1.45','requests','PyYAML','pandas','seaborn','scipy','pillow','tqdm','psutil','opencv-python-headless'],'detector_deps')
    script="""import sys,urllib.request
from pathlib import Path
sys.path.insert(0,str(Path.cwd()/'scripts'))
from setup_kaggle import pinned_repo
from vvcrl.data import DETECTOR_ID
from vvcrl.config import digest
tools=Path(sys.argv[1])
pinned_repo('https://github.com/ultralytics/yolov5.git',tools/'yolov5',DETECTOR_ID['source'])
w=tools/'yolov5n.pt'
if not w.exists():
    tmp=w.with_suffix('.download')
    urllib.request.urlretrieve('https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5n.pt',tmp)
    tmp.replace(w)
assert digest(w)==DETECTOR_ID['weights']
"""
    live([sys.executable,'-c',script,TOOLS],'detector_restore',REPO)
    verify_source();gate_check()

def show_progress():
    p=TRAIN/'PROGRESS.json'
    if not p.is_file():
        print('NO_CHECKPOINT_YET');return None
    r=json.loads(p.read_text());print('PROGRESS',json.dumps(r,indent=2))
    assert r['episodes']==2*r['iteration']
    assert sha256(TRAIN/r['checkpoint'])==r['checkpoint_sha256']
    for p in sorted((TRAIN/'validations').glob('*.json')):
        v=json.loads(p.read_text());print('VAL',p.stem,v['complete'],v['feasible'],v['macro_delta_quality'],v['macro_delta_utility'])
    return r


## Exact source and dataset

In [ ]:
BASE.mkdir(parents=True,exist_ok=True)
if not REPO.exists():live(['git','clone','--branch','research/d-series-adaptive-policy',REPOSITORY,REPO],'clone')
else:
    assert (REPO/'.git').is_dir()
    live([sys.executable,'-c',"from vvcrl.repository import verify_checkout; print('TRACKED_RAW_BYTES_PASS',verify_checkout('.'))"],'existing_source_verify',REPO)
live(['git','-C',REPO,'fetch','--no-tags','origin',SOURCE_COMMIT],'fetch')
live(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT],'checkout')
verify_source()
live([sys.executable,'-c','import torch; print(torch.__version__,torch.cuda.device_count()); assert torch.cuda.is_available() and torch.cuda.device_count()>=2'], 'gpu_check',REPO)
live([sys.executable,'-c',"from vvcrl.locked.data import VideoCatalog; import sys; c=VideoCatalog(sys.argv[1],mode='mini',seed=20261111); c.check_available(('train','val')); print('DATASET_PASS',len(c.selected['train']),len(c.selected['val']))",INPUT],'dataset_check',REPO)
print('FREE_GB',shutil.disk_usage(WORK).free/1e9)

## Import prior gate and exact binaries (no rebuild)

In [ ]:
gate_package=locate(GATE_NAME,PREFLIGHT_INPUT)
tool_package=locate(TOOLS_NAME,TOOLCHAIN_INPUT)
import_safe(gate_package,PREFLIGHT,'import_gate')
restore_toolchain(tool_package)
print('ORIGINAL_PREFLIGHT_AND_TOOLCHAIN_PASS')

## Training — gated and exact update target

In [ ]:
TARGET=10
resume=[]
if PARTIAL_STAGE_INPUT:
    p=safe_package(PARTIAL_STAGE_INPUT)
    assert p.is_file() and '_FAILED_' not in p.name
    import_safe(p,TRAIN,'import_partial')
    r=show_progress()
    assert r is not None and 0 < r['iteration'] <= 10
    resume=['--resume',TRAIN/'checkpoints/latest.pt']
else:
    assert not TRAIN.exists() or not any(TRAIN.iterdir())
    r=None
if resume:
    bind=json.loads((TRAIN/'TRAIN_BINDING.json').read_text())
    assert bind==json.loads((PREFLIGHT/'PREFLIGHT_GATE.json').read_text())['binding'], 'Resume binding differs from gate'
    protocol=json.loads((TRAIN/'RUN_PROTOCOL.json').read_text())
    assert protocol['seed']==SEED and protocol['val_every']==VAL_EVERY
current=r['iteration'] if r else 0
additional=TARGET-current
pending=r['validation_pending'] if r else False
assert 0<=additional<=10
verify_source();gate_check()
if additional or pending:
    remaining=SESSION_HOURS-(time.monotonic()-STARTED)/3600
    assert remaining>0.5,'Insufficient session time'
    live([sys.executable,'-u','-m','vvcrl','train-mini','--output',TRAIN,'--preflight',PREFLIGHT,
        '--updates',additional,'--val-every',VAL_EVERY,'--hours',remaining,*common_args(),'--start-mini-training',*resume],'training',REPO)
else:
    print('TARGET_ALREADY_COMPLETE')
    # Re-export an already-complete imported stage to its canonical output name.
    live([sys.executable,'-u','-c',"from vvcrl.artifacts import export_run; import sys; export_run(sys.argv[1],sys.argv[2])",TRAIN,WORK/(RUN10_NAME)],'export_complete',REPO)
r=show_progress()
package=WORK/(RUN10_NAME)
complete=bool(r and r['iteration']==TARGET and r['validation_pending'] is False and package.is_file())
print('STAGE_STATUS','COMPLETE' if complete else 'INCOMPLETE_RESUME_REQUIRED')
print('SAFE_PACKAGE',package)
if not complete:print('Do not proceed to the next stage. Preserve the latest non-FAILED SAFE package and set PARTIAL_STAGE_INPUT in a fresh session.')


## Output handoff
Save Version with the required files in Kaggle Output. A failed or partial run is not a completed stage. Keep the same source, environment, seed, variant and binary identities; never bypass a failed gate.